# Chapter 6 — Illustrative study (simulated)

Companion notebook for *Evaluating Large Language Models in High-Stakes Domains* (Costa, 2026). All data in this notebook are simulated; the generating process is in the code and described in the text.

In [ ]:
"""Figures and worked-example numbers for 'Evaluating Large Language Models in High-Stakes Domains'.
All data are analytic or simulated; generating processes are stated in the text.
Run:  python figures.py   (writes PNGs and results.json into ./figures)
"""
import os, json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize, minimize_scalar

OUT = os.path.join(os.getcwd(), "figures_out")
os.makedirs(OUT, exist_ok=True)

# palette (categorical, fixed order) and ink
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2,
    "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "legend.frameon": False, "figure.dpi": 200, "savefig.dpi": 200,
    "savefig.facecolor": "white",
})
results = {}
se = lambda x: x.std(ddof=1)/np.sqrt(len(x))
def wilson(x, n, z=1.96):
    p = x/n; c = (p + z*z/(2*n))/(1 + z*z/n)
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/(1 + z*z/n)
    return c-h, c+h
def clopper_upper(x, n, conf=0.95):
    return 1.0 if x == n else stats.beta.ppf(conf, x+1, n-x)


## Figure 5 + Tables 6.3/6.4: illustrative study (Ch. 6)

In [ ]:
rng = np.random.default_rng(2026)
models = ["Model 1", "Model 2", "Model 3", "Model 4"]
conds = ["API, item-by-item", "Chat, batched"]
n_items = 150
item_eff = rng.normal(0, 0.9, n_items)          # shared item difficulty (rubric)
safety_eff = rng.normal(0, 1.0, n_items)        # separate latent: how "dangerous" an item is
base = {"Model 1": 1.9, "Model 2": 1.6, "Model 3": 1.45, "Model 4": 0.7}
base_unsafe = {"Model 1": -3.9, "Model 2": -2.6, "Model 3": -2.2, "Model 4": -1.5}   # logit of P(unsafe)
cond_shift = {"API, item-by-item": 0.0, "Chat, batched": -0.5}
cond_unsafe = {"API, item-by-item": 0.0, "Chat, batched": 0.6}
score = {}; unsafe6 = {}; severe = {}
for cnd in conds:
    for m in models:
        lg = base[m] + cond_shift[cnd] + item_eff + rng.normal(0, 0.3, n_items)
        p = 1/(1+np.exp(-lg))
        crit = np.clip(p[:, None] + rng.normal(0, 0.15, (n_items, 4)), 0, 1)
        score[(cnd, m)] = crit.mean(1)
        pu = 1/(1+np.exp(-(base_unsafe[m] + cond_unsafe[cnd] + 0.9*safety_eff)))
        u = rng.binomial(1, pu); unsafe6[(cnd, m)] = u
        severe[(cnd, m)] = u * rng.binomial(1, 0.3, n_items)      # 30% of unsafe outputs are severe
tab = {}
for key, s in score.items():
    u, sv = unsafe6[key], severe[key]
    gated = s*(1-u)
    tab[key] = dict(mean=s.mean(), se=se(s), unsafe=u.mean(), n_sev=int(sv.sum()),
                    sev_ub=clopper_upper(int(sv.sum()), n_items), gated=gated.mean(), gated_se=se(gated))
rows6 = []
for m in models:
    cells = []
    for cnd in conds:
        t = tab[(cnd, m)]
        cells.append(f"{t['mean']:.3f} ({t['mean']-1.96*t['se']:.3f}–{t['mean']+1.96*t['se']:.3f}) | {100*t['unsafe']:.1f}% | {t['n_sev']} ({100*t['sev_ub']:.1f}%) | {t['gated']:.3f}")
    rows6.append(f"| {m} | " + " | ".join(cells) + " |")
results["ch6"] = {"CH6_TABLE": "\n".join(rows6)}
# paired comparisons: adjacent models within condition (6) + condition within model (4) = 10, Holm
comps = []
for cnd in conds:
    for a, b in [("Model 1", "Model 2"), ("Model 2", "Model 3"), ("Model 3", "Model 4")]:
        dd = score[(cnd, a)] - score[(cnd, b)]
        comps.append((f"{a} − {b}", cnd.split(",")[0], dd))
for m in models:
    dd = score[(conds[0], m)] - score[(conds[1], m)]
    comps.append((m, "API − Chat", dd))
pvals = np.array([stats.ttest_1samp(dd, 0).pvalue for _, _, dd in comps])
order = np.argsort(pvals); holm = np.empty_like(pvals); mcount = len(pvals)
running = 0.0
for rank, i in enumerate(order):
    adj = min(1.0, (mcount - rank)*pvals[i]); running = max(running, adj); holm[i] = running
rows_pd = []
for (lab_, ctx, dd), p_raw, p_h in zip(comps, pvals, holm):
    rows_pd.append(f"| {lab_} | {ctx} | {dd.mean():+.3f} ({dd.mean()-1.96*se(dd):+.3f}, {dd.mean()+1.96*se(dd):+.3f}) | {p_h:.3f}{'*' if p_h < 0.05 else ''} |")
results["ch6"]["PAIRED_TABLE"] = "\n".join(rows_pd)
results["ch6"]["N_SIG"] = str(int((holm < 0.05).sum()))
# a few named numbers for the prose
imin = int(np.argmin([abs(dd.mean()) for _, _, dd in comps])); dmin = comps[imin][2]
results["ch6"].update({
    "MIN_LABEL": f"{comps[imin][0]} ({comps[imin][1]})",
    "MIN_DIFF": f"{dmin.mean():+.3f} ({dmin.mean()-1.96*se(dmin):+.3f}, {dmin.mean()+1.96*se(dmin):+.3f})",
    "MIN_P": f"{holm[imin]:.3f}",
    "M1_API_SEV": f"{tab[(conds[0], 'Model 1')]['n_sev']}", "M1_API_SEV_UB": f"{100*tab[(conds[0], 'Model 1')]['sev_ub']:.1f}",
    "M1_CHAT_SEV": f"{tab[(conds[1], 'Model 1')]['n_sev']}", "M1_CHAT_SEV_UB": f"{100*tab[(conds[1], 'Model 1')]['sev_ub']:.1f}",
    "M1_API_UNSAFE": f"{100*tab[(conds[0], 'Model 1')]['unsafe']:.1f}", "M1_CHAT_UNSAFE": f"{100*tab[(conds[1], 'Model 1')]['unsafe']:.1f}",
    "M1_API_GATED": f"{tab[(conds[0], 'Model 1')]['gated']:.3f}",
})
# figure: (a) means with CIs, (b) paired differences with 95% CIs, Holm-significant marked
fig, ax = plt.subplots(1, 2, figsize=(7.4, 3.1), gridspec_kw={"width_ratios": [1, 1.15]})
ypos = np.arange(len(models))
for j, (cnd, col, mk) in enumerate(zip(conds, [C1, C2], ["o", "s"])):
    vals = [tab[(cnd, m)] for m in models]
    ax[0].errorbar([v["mean"] for v in vals], ypos + (0.15 if j else -0.15), xerr=[1.96*v["se"] for v in vals],
                   fmt=mk, color=col, ecolor=col, capsize=3, ms=6, lw=1.5, label=cnd)
ax[0].set_yticks(ypos); ax[0].set_yticklabels(models); ax[0].invert_yaxis()
ax[0].set_xlabel("Mean rubric score with 95% CI"); ax[0].set_xlim(0.4, 1.0)
ax[0].legend(loc="upper left", fontsize=7.5); ax[0].set_title("(a) Scores by model and condition", loc="left", fontsize=9, color=INK)
labels = [f"{l} ({c})" if c != "API − Chat" else f"{l}: API − Chat" for l, c, _ in comps]
means = np.array([dd.mean() for _, _, dd in comps]); halfs = np.array([1.96*se(dd) for _, _, dd in comps])
cols = [C1 if c == "API" else (C2 if c == "Chat" else C3) for _, c, _ in comps]
yy = np.arange(len(comps))
for i in range(len(comps)):
    ax[1].errorbar(means[i], yy[i], xerr=halfs[i], fmt="o", color=cols[i], ecolor=cols[i], capsize=3, ms=5, lw=1.5)
    if holm[i] < 0.05: ax[1].text(means[i]+halfs[i]+0.01, yy[i], "*", color=INK, va="center", fontsize=10)
ax[1].axvline(0, color=INK2, ls="--", lw=1)
ax[1].set_yticks(yy); ax[1].set_yticklabels(labels, fontsize=7); ax[1].invert_yaxis()
ax[1].set_xlabel("Paired difference, 95% CI (* Holm p < 0.05)")
ax[1].set_title("(b) Pre-specified paired comparisons", loc="left", fontsize=9, color=INK)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig5_illustrative.png")); plt.close(fig)

In [ ]:
import json
print(json.dumps(results, indent=2, ensure_ascii=False))